In [9]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler

In [10]:
# Load Dataset
df = pd.read_csv("dataset/raw_loan_dataset.csv")

print("="*60)
print("FIRST FIVE ROWS")
print("="*60)
print(df.head())

print("\n")

print("="*60)
print("DATA INFO")
print("="*60)
print(df.info())

print("\n")

print("="*60)
print("MISSING VALUES")
print("="*60)
print(df.isnull().sum())

FIRST FIVE ROWS
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


DATA INFO
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     str 

In [11]:
df["Income"] = (
    df["Income"]
    .astype(str)
    .str.replace("$","",regex=False)
    .str.replace(",","",regex=False)
)

df["LoanAmount"] = (
    df["LoanAmount"]
    .astype(str)
    .str.replace("$","",regex=False)
    .str.replace(",","",regex=False)
)

df["Income"] = pd.to_numeric(df["Income"], errors="coerce")
df["LoanAmount"] = pd.to_numeric(df["LoanAmount"], errors="coerce")

print(df[["Income","LoanAmount"]].head())

     Income  LoanAmount
0  108810.0     25800.0
1   96482.0     11200.0
2   28478.0     12100.0
3   25851.0      7000.0
4   38396.0     10700.0


In [12]:
# HasCollateral

df["HasCollateral"] = (
    df["HasCollateral"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["HasCollateral"] = df["HasCollateral"].replace({
    "yes":"Yes",
    "y":"Yes",
    "yse":"Yes",
    "no":"No",
    "n":"No"
})

print(df["HasCollateral"].value_counts(dropna=False))

HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64


In [13]:
# PreviousDefaults

df["PreviousDefaults"] = (
    df["PreviousDefaults"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["PreviousDefaults"] = df["PreviousDefaults"].replace({
    "yes":"Yes",
    "no":"No",
    "1":"Yes",
    "0":"No",
    "y":"Yes",
    "n":"No"
})

print(df["PreviousDefaults"].value_counts(dropna=False))

PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64


In [14]:
# Approved

df["Approved"] = (
    df["Approved"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Approved"] = df["Approved"].replace({
    "approved":"Yes",
    "yes":"Yes",
    "rejected":"No",
    "no":"No"
})

print(df["Approved"].value_counts(dropna=False))

Approved
Yes    68
No     35
Name: count, dtype: int64


In [15]:
numeric_columns = [
    "Income",
    "CreditScore",
    "EmploymentYears",
    "LoanAmount"
]

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

categorical_columns = [
    "HasCollateral",
    "PreviousDefaults",
    "Approved"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


In [16]:
print("Rows Before:", len(df))

df = df.drop_duplicates()

print("Rows After :", len(df))

Rows Before: 103
Rows After : 100


In [17]:
columns = [
    "Income",
    "CreditScore",
    "EmploymentYears",
    "LoanAmount"
]

for col in columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    df[col] = df[col].clip(lower, upper)

print("Outliers capped successfully.")

Outliers capped successfully.


In [18]:
mapping = {
    "Yes":1,
    "No":0
}

df["HasCollateral"] = df["HasCollateral"].map(mapping)

df["PreviousDefaults"] = df["PreviousDefaults"].map(mapping)

df["Approved"] = df["Approved"].map(mapping)

print(df["Approved"].value_counts())

Approved
1    66
0    34
Name: count, dtype: int64


In [19]:
print(df["Approved"].value_counts())

print()

print(df["Approved"].value_counts(normalize=True))

Approved
1    66
0    34
Name: count, dtype: int64

Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [20]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"]

df["IncomePerYearEmployed"] = (
    df["Income"] /
    (df["EmploymentYears"] + 1)
)

print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              1   
1   96482.0        524.0             15.0     11200.0              1   
2   28478.0        638.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              1   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.237111           51814.285714  
1                 0         1      0.116084            6030.125000  
2                 0         1      0.424889            4449.687500  
3                 0         1      0.270783            1389.838710  
4                 0         1      0.278675            3555.185185  


In [21]:
scaler = RobustScaler()

numeric_features = [
    "Income",
    "CreditScore",
    "EmploymentYears",
    "LoanAmount",
    "DebtToIncome",
    "IncomePerYearEmployed"
]

df[numeric_features] = scaler.fit_transform(df[numeric_features])

print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  


In [22]:
print(df.info())

print()

print(df.isnull().sum())

print()

print(df.head())

df.to_csv("clean_loan_dataset.csv", index=False)

print("\nDataset saved successfully!")

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.2 KB
None

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64

     Income  CreditScor